In [1]:
import pandas as pd
pd.set_option("display.max_columns",50)

In [2]:
df = pd.read_csv(r"C:\Users\User\OneDrive\Desktop\DA Projects\job-market-dashboard\data\raw\postings.csv")

In [3]:
print(df.shape)

(123849, 31)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [5]:
# transpose for better readability
print(df.head(1).T)

                                                                            0
job_id                                                                 921716
company_name                                            Corcoran Sawyer Smith
title                                                   Marketing Coordinator
description                 Job descriptionA leading real estate firm in N...
max_salary                                                               20.0
pay_period                                                             HOURLY
location                                                        Princeton, NJ
company_id                                                          2774458.0
views                                                                    20.0
med_salary                                                                NaN
min_salary                                                               17.0
formatted_work_type                                             

In [6]:
# Missing valuse as percentage
print((df.isna().mean() * 100).round(1).sort_values(ascending = False))

closed_time                   99.1
skills_desc                   98.0
med_salary                    94.9
remote_allowed                87.7
applies                       81.2
max_salary                    75.9
min_salary                    75.9
currency                      70.9
compensation_type             70.9
pay_period                    70.9
normalized_salary             70.9
posting_domain                32.3
application_url               29.6
formatted_experience_level    23.7
fips                          22.1
zip_code                      16.9
views                          1.4
company_id                     1.4
company_name                   1.4
listed_time                    0.0
work_type                      0.0
sponsored                      0.0
job_id                         0.0
expiry                         0.0
application_type               0.0
original_listed_time           0.0
formatted_work_type            0.0
location                       0.0
description         

In [7]:
# Duplicates
print("Exact duplicates:", df.duplicated().sum())

Exact duplicates: 0


In [8]:
# Cardinality: how many unique values per column
print(df.nunique().sort_values())

remote_allowed                     1
sponsored                          1
compensation_type                  1
application_type                   4
pay_period                         5
currency                           6
formatted_experience_level         6
work_type                          7
formatted_work_type                7
applies                          274
views                            684
closed_time                      698
med_salary                      1417
skills_desc                     2212
fips                            2270
posting_domain                  4443
min_salary                      4612
max_salary                      5321
zip_code                        6989
normalized_salary               7259
location                        8526
company_name                   24428
company_id                     24474
listed_time                    53231
expiry                         54851
original_listed_time           65036
title                          72521
a


| Issue | Column(s) | Evidence | Fix (phase) |
| :--- | :--- | :--- | :--- |
| Covers all job types | `title` | 123,849 rows, all industries | Filter to data and analytics titles (Phase 4, A.4) |
| Salary mostly missing, split across two forms | `min_salary`, `max_salary`, `med_salary` | 75.9% missing min/max, 94.9% missing med | `salary_mid` = median if present, else mean of min and max; leave blanks as NULL |
| Mixed pay periods and currencies | `pay_period`, `currency` | 5 pay periods, 6 currency values | Annualize, keep USD only |
| Experience level missing | `formatted_experience_level` | 23.7% missing | Fill with "Not specified" |
| Remote flag has one value | `remote_allowed` | 87.7% missing, only value is 1 | Fill blanks with 0 |
| Dates are epoch milliseconds | `original_listed_time` | float, no missing | Convert with `unit="ms"` |
| Location is "City, ST" or just "United States" | `location` | 8,526 unique values | Split into city and state; city blank when no state |
| Repeated descriptions | `description` | 107,827 unique of 123,842 | Check duplicates within your filtered data, then decide and document |
| Missing company and 7 missing descriptions | `company_name`, `description` | 1.4% and 7 rows | Keep rows, blank company; no skills for empty descriptions |
| Very large job IDs | `job_id` | up to about 3.9 billion | `BIGINT` in SQL Server, keep LinkedIn's ID |
| Unneeded columns | `urls`, `expiry`, `closed_time`, `zip_code`, `fips`, `posting_domain`, `sponsored`, `compensation_type` | no analytic use | Drop in Phase 4 |


In [9]:
df.sample(20)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
23179,3889748458,Liberty Healthcare and Rehabilitation Services,MEDICATION AIDE - LIBERTY COMMONS,\nLiberty Cares\nWith Compassion\nAt Liberty H...,NaN,NaN,"Wilmington, NC",53345529.0,5.0,NaN,NaN,Full-time,NaN,1.712667e+12,NaN,https://www.linkedin.com/jobs/view/3889748458/...,https://www.click2apply.net/Aezoa8TNXepR1HEygt...,OffsiteApply,1.715259e+12,1.712670e+12,NaN,NaN,1.712667e+12,www.click2apply.net,0,FULL_TIME,NaN,NaN,NaN,28401.0,37129.0
117230,3905882724,Spectrum Search Group,VP-Director-Oil & Gas Capital Markets,Vice President-Director: Oil & Gas Capital Mar...,300000.0,YEARLY,"New York, NY",10416987.0,3.0,NaN,250000.0,Full-time,NaN,1.713540e+12,NaN,https://www.linkedin.com/jobs/view/3905882724/...,NaN,ComplexOnsiteApply,1.716132e+12,NaN,Director,NaN,1.713540e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,275000.0,10001.0,36061.0
83531,3904071275,Keck Medicine of USC,Patient Access Liaison II - Access Center - Fu...,Serves as primary point of contact for access ...,NaN,NaN,"Alhambra, CA",899708.0,5.0,NaN,NaN,Full-time,NaN,1.713312e+12,NaN,https://www.linkedin.com/jobs/view/3904071275/...,https://usccareers.usc.edu/job/alhambra/patien...,OffsiteApply,1.716113e+12,NaN,Entry level,NaN,1.713521e+12,usccareers.usc.edu,0,FULL_TIME,NaN,NaN,NaN,91801.0,6037.0
91823,3904566209,BWise Trailers,Prepper,"Job Summary\n\nGrind, Sand, and Deburr metal s...",NaN,NaN,"Chambersburg, PA",11257573.0,1.0,NaN,NaN,Full-time,NaN,1.713550e+12,NaN,https://www.linkedin.com/jobs/view/3904566209/...,https://www.paycomonline.net/v4/ats/web.php/jo...,OffsiteApply,1.716142e+12,NaN,NaN,NaN,1.713550e+12,www.paycomonline.net,0,FULL_TIME,NaN,NaN,NaN,17201.0,42055.0
84451,3904360799,Circle K,Team Member,"Store 2746190: 4795 Miller Trunk Hwy, Hermanto...",NaN,NaN,"Hermantown, MN",12446.0,2.0,NaN,NaN,Part-time,NaN,1.713312e+12,NaN,https://www.linkedin.com/jobs/view/3904360799/...,https://workwithus.circlek.com/global/en/job/C...,OffsiteApply,1.715981e+12,NaN,Entry level,NaN,1.713389e+12,workwithus.circlek.com,0,PART_TIME,NaN,NaN,NaN,56248.0,27051.0
68047,3902758468,RK,HVAC Service Technician Lead (up to $5000 sign...,Description\n\nRK Company Overview\n\nAs a sec...,NaN,NaN,"Denver, CO",121711.0,5.0,NaN,NaN,Full-time,NaN,1.713394e+12,NaN,https://www.linkedin.com/jobs/view/3902758468/...,http://jobs.jobvite.com/rkmi/job/oRrZsfwt?__jv...,OffsiteApply,1.715987e+12,NaN,Mid-Senior level,NaN,1.713395e+12,jobs.jobvite.com,0,FULL_TIME,NaN,NaN,NaN,80202.0,8031.0
111408,3905392720,Tyson Foods,Forklift Operator - C Shift - Pine Bluff Freezer,Job Details:\n\nTuesday - Friday \n\nHours - 7...,NaN,NaN,"Pine Bluff, AR",166551.0,4.0,NaN,NaN,Full-time,NaN,1.713501e+12,NaN,https://www.linkedin.com/jobs/view/3905392720/...,https://tysonfoods.wd5.myworkdayjobs.com/TSN/j...,OffsiteApply,1.716094e+12,NaN,NaN,NaN,1.713502e+12,tysonfoods.wd5.myworkdayjobs.com,0,FULL_TIME,NaN,NaN,NaN,71601.0,5069.0
4830,3884925949,Beth Israel Lahey Health,Surgical ICU Nursing Assistant 36 hr days,"When you join the growing BILH team, you're no...",NaN,NaN,"Burlington, MA",28978143.0,4.0,NaN,NaN,Full-time,NaN,1.712349e+12,NaN,https://www.linkedin.com/jobs/view/3884925949/...,https://ad.doubleclick.net/ddm/clk/538489635;3...,OffsiteApply,1.714942e+12,NaN,Entry level,NaN,1.712350e+12,jobs.bilh.org,0,FULL_TIME,NaN,NaN,NaN,1803.0,25017.0
67817,3902756833,Citi,"SVP, Emerging Corporates Software Subsector He...",Job Id: 24734388\n\nThe Software Team Lead rep...,288000.0,YEARLY,"San Francisco, CA",11448.0,28.0,NaN,192000.0,Full-time,2.0,1.713312e+12,NaN,https://www.linkedin.com/jobs/view/3902756833/...,https://ad.doubleclick.net/ddm/trackcl

In [10]:
# 1.Salary logic
print("Rows with median and min/max",(df["med_salary"].notna() & df["min_salary"].notna() & df["max_salary"].notna()).sum())
print("pay period is filled but no salary",(df["pay_period"].notna() & df[["med_salary","min_salary","max_salary"]].isna().all(axis=1)).sum())
print(df.groupby("pay_period")[["med_salary","min_salary","max_salary"]].median())

Rows with median and min/max 0
pay period is filled but no salary 0
            med_salary  min_salary  max_salary
pay_period                                    
BIWEEKLY           NaN     54009.0     66976.0
HOURLY            21.0        25.0        32.0
MONTHLY         2364.5      5550.0      7350.0
WEEKLY             NaN      2099.0      2099.0
YEARLY         71905.0     87000.0    120000.0


In [11]:
# 2.Dates
d = pd.to_datetime(df["original_listed_time"],unit = "ms")
print(d.min(),d.max())
print(d.dt.to_period("M").value_counts().sort_index())

2023-12-05 21:08:53 2024-04-20 00:26:43
original_listed_time
2023-12         3
2024-01         7
2024-02         9
2024-03      1815
2024-04    122015
Freq: M, Name: count, dtype: int64


In [13]:
# 3.Category values
for c in ["formatted_experience_level","formatted_work_type","currency"]:
    print(df[c].value_counts(dropna=False),"\n")

formatted_experience_level
Mid-Senior level    41489
Entry level         36708
NaN                 29409
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64 

formatted_work_type
Full-time     98814
Contract      12117
Part-time      9696
Temporary      1190
Internship      983
Volunteer       562
Other           487
Name: count, dtype: int64 

currency
NaN    87776
USD    36058
EUR        6
CAD        3
BBD        2
AUD        2
GBP        2
Name: count, dtype: int64 



In [14]:
# 4. Location shape and look at real rows
print(df["location"].str.match(r"^[^,]+,\s*[A-Z]{2}$").mean())
df[["title","company_name","location","pay_period","min_salary","max_salary",
    "formatted_experience_level"]].sample(20, random_state=1)

0.8485332945764601


,title,company_name,location,pay_period,min_salary,max_salary,formatted_experience_level
109896,Trademark Attorney,Marina Sirras & Associates LLC,"New York, NY",YEARLY,225000.0,260000.0,Associate
15087,Delivery Driver / CDL A required / Seasonal,PepsiCo,"Alexandria, LA",HOURLY,NaN,NaN,NaN
101176,Senior Recruiter,The Armstrong Company,Memphis Metropolitan Area,NaN,NaN,NaN,NaN
106314,Order Fulfillment Coordinator,Allied OneSource,"Jacksonville, FL",NaN,NaN,NaN,Entry level
79678,Medical Assistant Urgent Care Per Diem,Carbon Health,"Beverly, MA",HOURLY,20.0,25.0,Entry level
91020,Endoscopy Technician,RUSH University Medical Center,"Chicago, IL",NaN,NaN,NaN,Entry level
53258,OT and DCS Team Lead,bp,"Chicago, IL",NaN,NaN,NaN,Mid-Senior level
107336,Production Supervisor - 2nd Shift,Tyson Foods,"Nashville, AR",NaN,NaN,NaN,NaN
48319,Associate Software Developer (Java/Scala),FlexTrade,"Milwaukee, WI",NaN,NaN,NaN,Associate
69185,Senior Information Technology Project Manager,Selby Jennings,"Boston, MA",NaN,NaN,NaN,Mid-Senior level
